## CSV 전체 구조 및 결측치 점검

이 노트북은 `data/` 폴더에 있는 모든 CSV 파일을 순회하며 데이터 구조와 결측치를 요약합니다. 출력된 표를 통해 어떤 파일이 비어 있거나 결측치가 많은지 빠르게 파악할 수 있습니다.

In [8]:
from pathlib import Path
import json
import pandas as pd

In [9]:
DATA_DIR = Path('../data')
csv_files = sorted(DATA_DIR.rglob('*.csv'))
print(f'총 {len(csv_files)}개의 CSV를 찾았습니다.')
for path in csv_files[:5]:
    print(' -', path)

총 90개의 CSV를 찾았습니다.
 - ../data/export_value.csv
 - ../data/gics/industry.csv
 - ../data/gics/industry_group.csv
 - ../data/gics/sector.csv
 - ../data/gics/sub_industry.csv


In [4]:
def profile_csv(path: Path) -> dict:
    """단일 CSV 파일을 로드하고 구조/결측치 정보를 반환합니다."""
    path = Path(path)
    rel = path.as_posix()
    result = {'path': rel}
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        result['error'] = str(exc)
        return result
    rows, cols = df.shape
    null_counts = df.isna().sum()
    column_details = []
    for col in df.columns:
        nulls = int(null_counts[col])
        column_details.append({
            'name': col,
            'dtype': str(df[col].dtype),
            'nulls': nulls,
            'null_pct': (nulls / rows * 100) if rows else 0.0
        })
    total_nulls = int(null_counts.sum())
    result.update({
        'rows': int(rows),
        'columns': int(cols),
        'columns_with_nulls': int((null_counts > 0).sum()),
        'total_null_cells': total_nulls,
        'total_null_pct': (total_nulls / (rows * cols) * 100) if rows * cols else 0.0,
        'column_details': column_details
    })
    return result

In [ ]:
profiles = [profile_csv(path) for path in csv_files]
print('프로파일링 완료:', len(profiles))

In [ ]:
summary_records = []
for entry in profiles:
    summary_records.append({
        'path': entry['path'],
        'rows': entry.get('rows'),
        'columns': entry.get('columns'),
        'columns_with_nulls': entry.get('columns_with_nulls'),
        'total_null_pct': entry.get('total_null_pct'),
        'error': entry.get('error')
    })
summary_df = pd.DataFrame(summary_records)
summary_df.head()

In [ ]:
def top_group(path: str) -> str:
    parts = path.split('/')
    if len(parts) >= 2 and parts[0] == 'data':
        return '/'.join(parts[:2])
    return path

group_stats = (
    summary_df.dropna(subset=['rows', 'columns'])
    .assign(group=lambda df: df['path'].apply(top_group))
    .groupby('group')
    .agg(
        files=('path', 'count'),
        avg_rows=('rows', 'mean'),
        avg_cols=('columns', 'mean'),
        avg_null_pct=('total_null_pct', 'mean')
    )
    .round({'avg_rows': 2, 'avg_cols': 2, 'avg_null_pct': 3})
)
group_stats

In [ ]:
price_minutely_df = summary_df[summary_df['path'].str.startswith('data/price_minutely')]
price_minutely_df[['total_null_pct']].describe()

In [ ]:
price_minutely_df.nlargest(5, 'total_null_pct')[['path', 'total_null_pct', 'columns_with_nulls']]

In [ ]:
OUTPUT_PATH = Path('analysis/csv_profile.json')
OUTPUT_PATH.write_text(json.dumps(profiles, indent=2, ensure_ascii=False))
print(f'요약 결과 저장: {OUTPUT_PATH}')